> Download and tidy up *real* data for training.

In [1]:
import pathlib
import os
import time

import torch
# import pandas as pd

import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.data
import trapallada.plot

So that the notebook is aware of modifications in the the `import`ed modules

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory where observations' data will be saved

In [3]:
download_directory =  pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
download_directory.mkdir(exist_ok=True)

That in which data will be saved

In [4]:
output_directory = pathlib.Path.cwd().parent / 'data'
assert output_directory.exists()

## Data for Javi's paper

In [5]:
javi_obs = '0605950101'
javi_obs_dir = trapallada.xsa.util.download_extract_filter(obs=javi_obs, output_directory=download_directory)
if javi_obs_dir is None:
    javi_obs_dir = download_directory / javi_obs
javi_obs_dir

found file "/home/manu/data/trapallada/0605950101.tar", skipping...
found directory "/home/manu/data/trapallada/0605950101", skipping...


PosixPath('/home/manu/data/trapallada/0605950101')

The downloaded files

In [6]:
# list(obs_dir.iterdir())
fits_files = list(javi_obs_dir.glob('*.FTZ'))
fits_files[:3]

[PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRCTSR802C.FTZ'),
 PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRCTSR8016.FTZ'),
 PosixPath('/home/manu/data/trapallada/0605950101/P0605950101PNS003SRCTSR801F.FTZ')]

In [7]:
df, skipped_files_list = trapallada.xsa.util.read_fits_files(fits_files)
print(f'len{skipped_files_list=}')
df.head()

lenskipped_files_list=[]


,P0605950101PNS003SRCTSR802C.FTZ,P0605950101PNS003SRCTSR8016.FTZ,P0605950101PNS003SRCTSR801F.FTZ,P0605950101PNS003SRCTSR8020.FTZ,P0605950101PNS003SRCTSR800F.FTZ,P0605950101PNS003SRCTSR802B.FTZ,P0605950101PNS003SRCTSR804C.FTZ,P0605950101PNS003SRCTSR8008.FTZ,P0605950101PNS003SRCTSR8018.FTZ,P0605950101PNS003SRCTSR8022.FTZ,...,P0605950101PNS003SRCTSR8064.FTZ,P0605950101PNS003SRCTSR8036.FTZ,P0605950101PNS003SRCTSR8030.FTZ,P0605950101PNS003SRCTSR8012.FTZ,P0605950101PNS003SRCTSR8021.FTZ,P0605950101PNS003SRCTSR801C.FTZ,P0605950101PNS003SRCTSR801A.FTZ,P0605950101PNS003SRCTSR8009.FTZ,P0605950101PNS003SRCTSR8002.FTZ,P0605950101PNS003SRCTSR8032.FTZ
TIME,,,,,,,,,,,,,,,,,,,,,
2009-06-03 08:10:33.517454,0.000000,0.0,-0.030024,0.0,0.000000,0.000000,0.0,0.837148,0.000000,0.0,...,0.00000,0.0,-0.02643,0.000000,0.0,0.0,-0.00934,-0.125560,0.808501,0.000000
2009-06-03 08:10:34.977454,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,-0.03712,0.0,0.00000,0.000000,0.0,0.0,0.00000,0.000000,0.771650,0.000000
2009-06-03 08:10:36.437454,0.000000,0.0,0.000000,0.0,-0.050389,-0.020843,0.0,0.000000,0.000000,0.0,...,0.00000,0.0,0.00000,0.000000,0.0,0.0,0.00000,0.000000,0.765734,0.000000
2009-06-03 08:10:37.897454,-0.010277,0.0,0.000000,0.0,0.000000,0.000000,0.0,-0.090626,-0.010292,0.0,...,0.00000,0.0,0.00000,0.000000,0.0,0.0,0.00000,0.778995,0.000000,0.000000
2009-06-03 08:10:39.357454,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,...,0.00000,0.0,0.00000,-0.104794,0.0,0.0,0.00000,0.000000,0.778009,-0.021274


We have `number of rows` measurements for each one of the `number of columns` signals

In [8]:
df.shape

(31805, 49)

The period is

In [9]:
df.index[1] - df.index[0]

Timedelta('0 days 00:00:01.460000')

Signals are *binned* so that a bunch of samples/measurements are added up together

In [10]:
bin_size = 3
binned_df = trapallada.xsa.util.bin_fits_df(df, bin_size=bin_size)
binned_df

,P0605950101PNS003SRCTSR802C.FTZ,P0605950101PNS003SRCTSR8016.FTZ,P0605950101PNS003SRCTSR801F.FTZ,P0605950101PNS003SRCTSR8020.FTZ,P0605950101PNS003SRCTSR800F.FTZ,P0605950101PNS003SRCTSR802B.FTZ,P0605950101PNS003SRCTSR804C.FTZ,P0605950101PNS003SRCTSR8008.FTZ,P0605950101PNS003SRCTSR8018.FTZ,P0605950101PNS003SRCTSR8022.FTZ,...,P0605950101PNS003SRCTSR8064.FTZ,P0605950101PNS003SRCTSR8036.FTZ,P0605950101PNS003SRCTSR8030.FTZ,P0605950101PNS003SRCTSR8012.FTZ,P0605950101PNS003SRCTSR8021.FTZ,P0605950101PNS003SRCTSR801C.FTZ,P0605950101PNS003SRCTSR801A.FTZ,P0605950101PNS003SRCTSR8009.FTZ,P0605950101PNS003SRCTSR8002.FTZ,P0605950101PNS003SRCTSR8032.FTZ
TIME,,,,,,,,,,,,,,,,,,,,,
2009-06-03 08:10:37.897454,-0.010277,0.000000,0.000000,0.000000,-0.050389,-0.020843,0.000000,-0.090626,-0.010292,0.00000,...,-0.037120,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.778995,1.537384,0.000000
2009-06-03 08:10:42.277454,0.000000,-0.024786,0.000000,0.000000,0.000000,-0.027165,-0.030863,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.00000,-0.104794,0.000000,0.000000,0.000000,0.000000,3.108174,-0.021274
2009-06-03 08:10:46.657454,-0.010199,0.784629,-0.024984,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,-0.011943,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,1.562582,0.000000
2009-06-03 08:10:51.037454,-0.020730,0.000000,0.000000,0.000000,0.000000,-0.020932,-0.023781,0.780824,-0.010478,-0.02778,...,-0.037124,-0.011736,0.00000,0.000000,0.000000,-0.011793,0.000000,0.000000,2.079360,-0.021407
2009-06-03 08:10:55.417454,-0.020806,0.000000,0.000000,-0.023497,0.000000,-0.021987,-0.024980,0.774934,0.000000,0.00000,...,-0.038198,0.000000,0.00000,0.000000,0.000000,-0.023557,-0.008848,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2009-06-03 21:04:08.377454,0.000000,0.000000,0.000000,-0.026902,-0.055202,-0.022834,-0.025942,0.000000,-0.020772,0.00000,...,0.000000,0.000000,0.00000,0.000000,0.000000,-0.026543,0.000000,0.000000,0.000000,-0.021423
2009-06-03 21:04:12.757454,0.000000,0.000000,0.000000,0.000000,-0.050222,-0.020774,-0.023602,0.676927,-0.020948,0.00000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.519583,0.000000
2009-06-03 21:04:17.137454,0.000000,0.000000,0.000000,0.000000,-0.100765,-0.020841,-0.023770,-0.180045,-0.010366,0.00000,...,-0.037390,0.000000,0.00000,0.000000,0.000000,-0.023771,0.000000,-0.114098,0.777761,0.000000


The period is now

In [11]:
binned_df.index[1] - binned_df.index[0]

Timedelta('0 days 00:00:04.380000')

Signals might have `NaN`s

In [12]:
signals = binned_df.values.T
np.isnan(signals).sum()
# np.argwhere(np.isnan(signals))

49

if there are `NaN`s, they are fixed by interpolation

In [13]:
if binned_df.isna().any().any():
    binned_df = binned_df.interpolate(method='linear', axis=0)

Output file...

In [14]:
output_file = (output_directory / f'{javi_obs}_bin_size_{bin_size}').with_suffix('.pt')
output_file

PosixPath('/home/manu/Sync/git/trapallada/data/0605950101_bin_size_3.pt')

...to save the signals (notice we add an extra *dummy* axis)

In [15]:
# df.to_parquet(output_file)
torch.save(np.expand_dims(binned_df.values.T, 1), output_file)

A `SignalsDataset` is instantiated

In [16]:
ds = trapallada.data.SignalsDataset(output_file, metadata_filename=None, normalize=True)
len(ds)

49

A `DataFrame` is built out of the `Dataset` in order to make the plot (notice columns are renamed to avoid `.`s). `trapallada.plot.signals_dataset` cannot be called since this is not a *fullsome* `SignalsDataset`(no *metadata* `DataFrame`).

In [17]:
trapallada.plot.signals_dataset(ds, columns=[pathlib.Path(c).stem for c in df.columns], n_signals=None)

alt.Chart(...)

In [18]:
n_samples_to_plot = 2_000
sample_signal = binned_df.iloc[:n_samples_to_plot, 0].values
trapallada.plot.signal(sample_signal)

alt.Chart(...)

# Dataset

**All** the observation in XSA whose duration is in the given interval are downloaded

In [19]:
# trapallada.xsa.util.download_observations(download_directory=download_directory, duration_interval=(35_000, 45_000))

*Sub*directories in the observations directory

In [20]:
obs_dirs = [e for e in download_directory.iterdir() if e.is_dir()]
obs_dirs[:5]

[PosixPath('/home/manu/data/trapallada/0093200101'),
 PosixPath('/home/manu/data/trapallada/0651460101'),
 PosixPath('/home/manu/data/trapallada/0300810101'),
 PosixPath('/home/manu/data/trapallada/0673190201'),
 PosixPath('/home/manu/data/trapallada/0504810601')]

Files (sources) in a *particular directory* are ranked.

In [21]:
i_obs_dir = 5
n_sources_per_obs = 3
# ranked_files_df = trapallada.xsa.util.rank_obs_files(obs_dirs[i_obs_dir]).iloc[:n_sources_per_obs]['full path']
ranked_files_df = trapallada.xsa.util.rank_obs_files(javi_obs_dir).iloc[:n_sources_per_obs]['full path']
ranked_files_df

28    /home/manu/data/trapallada/0605950101/P0605950...
47    /home/manu/data/trapallada/0605950101/P0605950...
36    /home/manu/data/trapallada/0605950101/P0605950...
Name: full path, dtype: object

The top `n_sources_per_obs` FITs files are read into a `DataFrame`

In [22]:
fits_df, skipped_files_list = trapallada.xsa.util.read_fits_files(ranked_files_df.tolist())
fits_df.head()

,P0605950101PNS003SRCTSR8001.FTZ,P0605950101PNS003SRCTSR8002.FTZ,P0605950101PNS003SRCTSR8003.FTZ
TIME,,,
2009-06-03 08:10:33.517454,0.878811,0.808501,0.808501
2009-06-03 08:10:34.977454,0.785685,0.771650,0.771650
2009-06-03 08:10:36.437454,0.000000,0.765734,0.000000
2009-06-03 08:10:37.897454,0.000000,0.000000,0.000000
2009-06-03 08:10:39.357454,0.000000,0.778009,0.683218


In [23]:
bin_size = 63
# bin_size = 32
binned_df = trapallada.xsa.util.bin_fits_df(fits_df, bin_size=bin_size)
binned_df

,P0605950101PNS003SRCTSR8001.FTZ,P0605950101PNS003SRCTSR8002.FTZ,P0605950101PNS003SRCTSR8003.FTZ
TIME,,,
2009-06-03 08:12:05.497454,34.577186,46.343955,7.729602
2009-06-03 08:13:37.477454,28.529714,36.475717,4.943364
2009-06-03 08:15:09.457454,48.805367,25.931725,14.755459
2009-06-03 08:16:41.437454,39.296789,39.233786,12.232534
2009-06-03 08:18:13.417454,36.757868,31.612971,13.566273
...,...,...,...
2009-06-03 20:57:03.517454,22.408824,30.035924,27.805105
2009-06-03 20:58:35.497454,29.378926,31.476628,27.736561
2009-06-03 21:00:07.477454,29.943388,27.180789,32.544844


In [24]:
trapallada.plot.df(binned_df, columns=[pathlib.Path(e).stem for e in binned_df.columns], index_type='T')

alt.Chart(...)

In [25]:
# fits_df, skipped_files_list = trapallada.xsa.util.read_fits_files(fits_files)
# fits_df.head()